<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Quantz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Quantz — Universal Export & Validation Pipeline

Exports every deployable checkpoint to FP32 ONNX + INT8 QDQ ONNX and validates on-device accuracy.

**Sources (auto-discovered — no manual list needed):**
- Seed baselines — hardcoded, always included
- Pruning checkpoints — loaded from `pruning_combined_records.json` (quant_ok = True only)
- KD checkpoints — loaded from `kd_combined_records.json` (quant_ok = True only)

**Per-run pipeline:**
1. Load checkpoint → evaluate FP32 val accuracy
2. Export FP32 ONNX → calibrate → INT8 QDQ ONNX
3. Compute NSE (FP32 vs INT8 logits)
4. Compute local INT8 accuracy on shared test set
5. Save record to `quantz_records.json` on Drive

**Resume support:** re-running skips already-exported runs automatically.  
STM32 on-device results are filled in separately after CLI runs.

**Never change** `SHARED_DIR` — all STM32 CLI commands must use the same `test_input.npz`.

In [1]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
# ── Imports ──────────────────────────────────────────────────────────
import json, math, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.3 MB/s eta 0:00:00
Device: cuda


In [3]:
# ── Config ───────────────────────────────────────────────────────────
SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")
SHARED_DIR = EXPORT_DIR / "shared"

# ── Source JSON records from upstream pipelines ───────────────────────
PRUNING_RECORDS_JSON = f"{SAVE_DIR}/pruning_combined_records.json"
KD_RECORDS_JSON      = f"{SAVE_DIR}/kd_combined_records.json"

# ── This pipeline's own records ───────────────────────────────────────
QUANTZ_RECORDS_JSON  = f"{SAVE_DIR}/quantz_records.json"

# ── NSE threshold (must match upstream pipelines) ─────────────────────
NSE_THRESHOLD = 0.95

# ── Seed baselines — always included regardless of upstream records ───
SEED_RUNS = [
    {"name": "mobilenetv2_seed_74", "arch": "mobilenetv2",
     "ckpt": f"{SAVE_DIR}/mobilenetv2_seed_74.pth"},
    {"name": "mobilenetv3_seed_74", "arch": "mobilenetv3",
     "ckpt": f"{SAVE_DIR}/mobilenetv3_seed_74.pth"},
]

print("✅ Config loaded")

✅ Config loaded


In [4]:
# ── Helpers ──────────────────────────────────────────────────────────

def build_model(arch):
    if arch == "mobilenetv2": return VWW_MobileNetV2().to(device)
    if arch == "mobilenetv3": return VWW_MobileNetV3().to(device)
    raise ValueError(f"Unknown arch: {arch!r}")


# AFTER (fixed — 100 non-person + 100 person, balanced)
def generate_shared_test_files(n_per_class=100):
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    inp_p = SHARED_DIR / "test_input.npz"
    lbl_p = SHARED_DIR / "test_labels.npz"
    if inp_p.exists() and lbl_p.exists():
        print(f"    ⏭️  Shared test files exist")
        return inp_p, lbl_p
    per_class = {0: [], 1: []}
    for x, y in test_loader:
        lbl = int(y.item())
        if len(per_class[lbl]) < n_per_class:
            per_class[lbl].append(x.numpy().astype("float32")[0])
        if all(len(v) == n_per_class for v in per_class.values()):
            break
    inputs = per_class[0] + per_class[1]
    labels = [0] * n_per_class + [1] * n_per_class
    np.savez(inp_p, input=np.stack(inputs))
    np.savez(lbl_p, label=np.array(labels, dtype="int32"))
    print(f"    ✅ Shared test files saved ({n_per_class*2} samples, balanced)")
    return inp_p, lbl_p


def export_onnx(model, path):
    if Path(path).exists():
        print(f"    ⏭️  FP32 ONNX exists"); return
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX saved")


def save_calib_npz(path, n=200):
    if Path(path).exists():
        print(f"    ⏭️  Calib data exists"); return
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    np.savez(path, input=np.stack(xs))
    print(f"    ✅ Calib data saved ({n} samples)")


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path, calib_path, int8_path):
    if Path(int8_path).exists():
        print(f"    ⏭️  INT8 ONNX exists"); return
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX saved")


def compute_nse(fp32_path, int8_path, input_npz):
    inputs = np.load(input_npz)["input"]
    s32 = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    s8  = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    fp32_outs, int8_outs = [], []
    for i in range(len(inputs)):
        sample = inputs[i:i+1]
        fp32_outs.append(s32.run(["logits"], {"input": sample})[0][0])
        int8_outs.append(s8.run(["logits"],  {"input": sample})[0][0])
    fp32_outs = np.array(fp32_outs)
    int8_outs = np.array(int8_outs)
    num = np.sum((fp32_outs - int8_outs) ** 2)
    den = np.sum((fp32_outs - fp32_outs.mean()) ** 2)
    return float(1 - num / den)


def compute_local_int8_accuracy(int8_path, input_npz, labels_npz):
    sess   = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    inputs = np.load(input_npz)["input"]
    labels = np.load(labels_npz)["label"].astype("int64")
    preds  = [int(np.argmax(sess.run(["logits"], {"input": inputs[i:i+1]})[0][0]))
              for i in range(len(inputs))]
    return float((np.array(preds) == labels).mean() * 100)


def compute_stm32_accuracy(labels_npz, outputs_npz,
                           key="c_outputs_1", num_classes=2):
    labels = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key]
    if raw.size != len(labels) * num_classes:
        raise ValueError(f"Size mismatch: {raw.size} vs {len(labels)*num_classes}")
    preds  = np.argmax(raw.reshape(len(labels), num_classes), axis=1)
    return float((preds == labels).mean() * 100)


print("✅ Helpers loaded")

✅ Helpers loaded


In [5]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation
✅ Dataset ready


In [6]:
# ── Build run list from upstream JSON records ────────────────────────
# Auto-discovers deployable checkpoints from pruning + KD pipelines.
# Falls back gracefully if a JSON doesn't exist yet.

RUNS = []

# 1. Seed baselines — always included
for r in SEED_RUNS:
    RUNS.append({
        "name"   : r["name"],
        "arch"   : r["arch"],
        "ckpt"   : r["ckpt"],
        "source" : "baseline",
    })

# 2. Pruning checkpoints — quant_ok = True only
if os.path.exists(PRUNING_RECORDS_JSON):
    with open(PRUNING_RECORDS_JSON) as f:
        pruning_records = json.load(f)
    added = 0
    for r in pruning_records:
        if not r.get("quant_ok", False):
            continue
        model      = r["model"]
        prune_type = r["prune_type"]
        pct        = r["target_%"]
        arch       = "mobilenetv2" if "V2" in model else "mobilenetv3"
        name       = f"{arch}_{prune_type}_{pct}pct"
        ckpt       = r.get("ckpt", f"{SAVE_DIR}/{name}.pth")
        RUNS.append({
            "name"   : name,
            "arch"   : arch,
            "ckpt"   : ckpt,
            "source" : f"pruning ({prune_type} {pct}%)",
        })
        added += 1
    print(f"✅ Loaded {added} deployable pruning checkpoints from records")
else:
    print(f"⚠️  Pruning records not found: {PRUNING_RECORDS_JSON}")
    print(f"   Run Model_Pruning_Combined first, or add checkpoints manually below.")

# 3. KD checkpoints — quant_ok = True only
if os.path.exists(KD_RECORDS_JSON):
    with open(KD_RECORDS_JSON) as f:
        kd_records = json.load(f)
    added = 0
    for r in kd_records:
        if not r.get("quant_ok", False):
            continue
        run_key  = r["run_key"]
        student  = r["student"]
        arch     = "mobilenetv2" if "V2" in student else "mobilenetv3"
        ckpt     = r.get("ckpt", f"{SAVE_DIR}/{run_key}.pth")
        RUNS.append({
            "name"   : run_key,
            "arch"   : arch,
            "ckpt"   : ckpt,
            "source" : f"kd ({r['teacher']} → {student} {r['init']})",
        })
        added += 1
    print(f"✅ Loaded {added} deployable KD checkpoints from records")
else:
    print(f"⚠️  KD records not found: {KD_RECORDS_JSON}")
    print(f"   Run Model_KD_Combined first, or add checkpoints manually below.")

# 4. Manual overrides — add extra checkpoints here if needed
# RUNS.append({
#     "name"   : "my_custom_model",
#     "arch"   : "mobilenetv2",
#     "ckpt"   : f"{SAVE_DIR}/my_custom_model.pth",
#     "source" : "manual",
# })

# Create export dirs
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for r in RUNS:
    r["out_dir"] = EXPORT_DIR / r["name"]
    r["out_dir"].mkdir(parents=True, exist_ok=True)

# Pre-flight
print(f"\n{'─'*70}")
print(f"Total runs: {len(RUNS)}")
print(f"{'─'*70}")
for r in RUNS:
    ok = "✅" if Path(r["ckpt"]).exists() else "❌ MISSING"
    print(f"  {ok}  {r['name']:<40}  [{r['source']}]")

missing = [r for r in RUNS if not Path(r["ckpt"]).exists()]
if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) missing — those runs will be skipped.")

✅ Loaded 7 deployable pruning checkpoints from records
✅ Loaded 8 deployable KD checkpoints from records

──────────────────────────────────────────────────────────────────────
Total runs: 17
──────────────────────────────────────────────────────────────────────
  ✅  mobilenetv2_seed_74                       [baseline]
  ✅  mobilenetv3_seed_74                       [baseline]
  ✅  mobilenetv2_unstructured_10pct            [pruning (unstructured 10%)]
  ✅  mobilenetv2_unstructured_20pct            [pruning (unstructured 20%)]
  ✅  mobilenetv2_unstructured_30pct            [pruning (unstructured 30%)]
  ✅  mobilenetv3_unstructured_10pct            [pruning (unstructured 10%)]
  ✅  mobilenetv3_unstructured_20pct            [pruning (unstructured 20%)]
  ✅  mobilenetv3_unstructured_30pct            [pruning (unstructured 30%)]
  ✅  mobilenetv3_structured_3pct               [pruning (structured 3%)]
  ✅  vgg_mv2_ft                                [kd (vgg → MobileNetV2 ft)]
  ✅  vgg_mv2_scra

In [7]:
# ── Shared test files — generated once, reused by all runs ───────────
print("[Shared test set]")
SHARED_INPUT_NPZ, SHARED_LABELS_NPZ = generate_shared_test_files(n_per_class=100)
print(f"  Input : {SHARED_INPUT_NPZ}")
print(f"  Labels: {SHARED_LABELS_NPZ}")

[Shared test set]
    ✅ Shared test files saved (200 samples, balanced)
  Input : /content/drive/My Drive/stm32-thesis/exports/shared/test_input.npz
  Labels: /content/drive/My Drive/stm32-thesis/exports/shared/test_labels.npz


In [8]:
# ── Main export + validation loop ───────────────────────────────────
# Resume: loads existing records, skips completed runs.
# Saves progress after every run — safe to interrupt and re-run.

# ── Load existing progress ────────────────────────────────────────────
if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)
    print(f"✅ Loaded {len(records)} existing records")
    for r in records:
        print(f"   ↩️  {r['name']:<40}  NSE: {r['NSE']}  "
              f"local_int8: {r['local_int8_acc_%']}%")
else:
    records = []
    print("No existing records — starting fresh")

print()

def _already_done(name):
    return any(r["name"] == name for r in records)

for run in RUNS:
    name    = run["name"]
    arch    = run["arch"]
    ckpt    = run["ckpt"]
    out_dir = run["out_dir"]
    source  = run["source"]

    # ── Skip if already exported ──────────────────────────────────────
    if _already_done(name):
        print(f"⏭️  {name} — already done, skipping")
        continue

    print(f"\n{'═'*65}")
    print(f"  {name}  [{source}]")
    print(f"{'═'*65}")

    if not Path(ckpt).exists():
        print(f"  ❌ Checkpoint missing, skipping: {ckpt}")
        continue

    # ── [1] Load + eval ───────────────────────────────────────────────
    model = build_model(arch)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    val_acc = evaluate(model, val_loader, device)
    print(f"  FP32 val acc : {val_acc*100:.2f}%")

    # ── [2] FP32 ONNX ─────────────────────────────────────────────────
    fp32_path  = out_dir / "model_fp32.onnx"
    calib_path = out_dir / "calib_train.npz"
    int8_path  = out_dir / "model_int8_qdq.onnx"

    export_onnx(model, fp32_path)
    save_calib_npz(calib_path)

    # ── [3] INT8 QDQ ──────────────────────────────────────────────────
    try:
        quantize_int8(fp32_path, calib_path, int8_path)
        nse           = compute_nse(fp32_path, int8_path, SHARED_INPUT_NPZ)
        local_int8    = compute_local_int8_accuracy(int8_path,
                                                    SHARED_INPUT_NPZ,
                                                    SHARED_LABELS_NPZ)
        nse_ok        = nse >= NSE_THRESHOLD
        quant_error   = None
    except Exception as e:
        nse         = float("nan")
        local_int8  = float("nan")
        nse_ok      = False
        quant_error = str(e)
        print(f"    ⚠️  Quantisation failed: {e}")

    gate = "✅ deploy" if nse_ok else "⛔ quant failed"
    print(f"  NSE          : {nse:.4f}  {gate}")
    print(f"  Local INT8   : {local_int8:.2f}%  "
          f"(200-sample estimate, not comparable to FP32 val)")

    records.append({
        "name"            : name,
        "arch"            : arch,
        "source"          : source,
        "fp32_val_acc_%"  : round(val_acc * 100, 2),
        "local_int8_acc_%": round(local_int8, 2) if not math.isnan(local_int8) else None,
        "NSE"             : round(nse, 4) if not math.isnan(nse) else None,
        "nse_ok"          : nse_ok,
        "quant_error"     : quant_error,
        "fp32_path"       : str(fp32_path),
        "int8_path"       : str(int8_path) if nse_ok else None,
        "out_dir"         : str(out_dir),
        "stm32_fp32_acc_%": None,   # filled in by STM32 cell
        "stm32_int8_acc_%": None,   # filled in by STM32 cell
    })

    # ── Save progress after every run ─────────────────────────────────
    with open(QUANTZ_RECORDS_JSON, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  💾 Progress saved ({len(records)} records total)")

    del model

print("\n\n✅ Export loop complete.")
print(f"   Records JSON: {QUANTZ_RECORDS_JSON}")

No existing records — starting fresh


═════════════════════════════════════════════════════════════════
  mobilenetv2_seed_74  [baseline]
═════════════════════════════════════════════════════════════════
  FP32 val acc : 78.40%


/tmp/ipykernel_524/2083203892.py:37: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


    ✅ FP32 ONNX saved


    ✅ Calib data saved (200 samples)


    ✅ INT8 QDQ ONNX saved
  NSE          : 0.9777  ✅ deploy
  Local INT8   : 75.00%  (200-sample estimate, not comparable to FP32 val)
  💾 Progress saved (1 records total)

═════════════════════════════════════════════════════════════════
  mobilenetv3_seed_74  [baseline]
═════════════════════════════════════════════════════════════════
  FP32 val acc : 79.13%
    ✅ FP32 ONNX saved


...

    ✅ INT8 QDQ ONNX saved
  NSE          : 0.9830  ✅ deploy
  Local INT8   : 80.50%  (200-sample estimate, not comparable to FP32 val)
  💾 Progress saved (15 records total)

═════════════════════════════════════════════════════════════════
  resnet_mv3_ft  [kd (resnet → MobileNetV3 ft)]
═════════════════════════════════════════════════════════════════
  FP32 val acc : 79.13%
    ✅ FP32 ONNX saved


    ✅ Calib data saved (200 samples)


    ✅ INT8 QDQ ONNX saved
  NSE          : 0.9888  ✅ deploy
  Local INT8   : 79.50%  (200-sample estimate, not comparable to FP32 val)
  💾 Progress saved (16 records total)

═════════════════════════════════════════════════════════════════
  resnet_mv3_scratch  [kd (resnet → MobileNetV3 scratch)]
═════════════════════════════════════════════════════════════════
  FP32 val acc : 79.27%
    ✅ FP32 ONNX saved


    ✅ Calib data saved (200 samples)


    ✅ INT8 QDQ ONNX saved
  NSE          : 0.9902  ✅ deploy
  Local INT8   : 83.50%  (200-sample estimate, not comparable to FP32 val)
  💾 Progress saved (17 records total)


✅ Export loop complete.
   Records JSON: /content/drive/My Drive/stm32-thesis/checkpoints/quantz_records.json


In [9]:
# ── Results table ────────────────────────────────────────────────────
# Reloads from JSON — works independently across sessions.

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)

df = pd.DataFrame(records)

W = 100
print("=" * W)
print("PIPELINE_QUANTZ — CONSOLIDATED RESULTS")
print("=" * W)

cols = ["name", "source", "fp32_val_acc_%", "local_int8_acc_%", "NSE", "nse_ok"]
# Note: local_int8_acc_% is on 200 calibration samples, fp32_val_acc_% is on 1500 val samples
# — these are different sets so no delta is shown. Use STM32 cell for apples-to-apples INT8 comparison.
print(df[cols].to_string(index=False))
print("=" * W)

# ── NSE summary ───────────────────────────────────────────────────────
print("\nNSE gate summary:")
for _, r in df.iterrows():
    gate    = "✅ PASS" if r["nse_ok"] else "⛔ FAIL"
    nse_str = f"{r['NSE']:.4f}" if r["NSE"] is not None else "ERROR"
    print(f"  {r['name']:<42}  NSE={nse_str}  {gate}")

# ── STM32 results (if available) ──────────────────────────────────────
stm32_cols = ["stm32_fp32_acc_%", "stm32_int8_acc_%"]
has_stm32  = df[stm32_cols].notna().any().any()
if has_stm32:
    print(f"\nSTM32 on-device results:")
    print(f"  {'Name':<42} {'FP32':>8}  {'INT8':>8}  {'Delta':>8}")
    print(f"  {'─'*72}")
    for _, r in df.iterrows():
        fp32 = f"{r['stm32_fp32_acc_%']:.2f}%" if r["stm32_fp32_acc_%"] is not None else "pending"
        int8 = f"{r['stm32_int8_acc_%']:.2f}%" if r["stm32_int8_acc_%"] is not None else "pending"
        if r["stm32_fp32_acc_%"] and r["stm32_int8_acc_%"]:
            delta = f"{r['stm32_int8_acc_%'] - r['stm32_fp32_acc_%']:+.2f}%"
        else:
            delta = "—"
        print(f"  {r['name']:<42} {fp32:>8}  {int8:>8}  {delta:>8}")
else:
    print("\nSTM32 on-device results: not yet available (run STM32 CLI, then Cell 12)")

PIPELINE_QUANTZ — CONSOLIDATED RESULTS
                          name                            source  fp32_val_acc_%  local_int8_acc_%    NSE  nse_ok
           mobilenetv2_seed_74                          baseline           78.40              75.0 0.9777    True
           mobilenetv3_seed_74                          baseline           79.13              81.0 0.9931    True
mobilenetv2_unstructured_10pct        pruning (unstructured 10%)           78.93              75.5 0.9250   False
mobilenetv2_unstructured_20pct        pruning (unstructured 20%)           78.53              77.0 0.9843    True
mobilenetv2_unstructured_30pct        pruning (unstructured 30%)           78.27              75.0 0.9799    True
mobilenetv3_unstructured_10pct        pruning (unstructured 10%)           78.67              80.0 0.9939    True
mobilenetv3_unstructured_20pct        pruning (unstructured 20%)           78.27              82.5 0.9926    True
mobilenetv3_unstructured_30pct        pruning (un

## STM32 CLI Commands

All models use the **same** `test_input.npz` from `exports/shared/`. Never change this path.

```bash
# Template — replace <name> with the run name (e.g. mobilenetv2_seed_74)

# FP32
stedgeai validate -m exports/<name>/model_fp32.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_fp32.npz

# INT8
stedgeai validate -m exports/<name>/model_int8_qdq.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_int8.npz
```

Run the cell below after copying `.npz` outputs back to Drive.

In [10]:
# ── STM32 on-device results ──────────────────────────────────────────
# Run this cell after STM32 CLI output .npz files are copied back to Drive.
# Updates quantz_records.json with on-device accuracy — safe to re-run
# as results come in incrementally.

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)

updated = 0
for r in records:
    out_dir   = Path(r["out_dir"])
    fp32_out  = out_dir / "output_fp32.npz"
    int8_out  = out_dir / "output_int8.npz"

    changed = False

    if fp32_out.exists() and r["stm32_fp32_acc_%"] is None:
        try:
            acc = compute_stm32_accuracy(SHARED_LABELS_NPZ, fp32_out)
            r["stm32_fp32_acc_%"] = round(acc, 2)
            print(f"  ✅ {r['name']:<42}  STM32 FP32: {acc:.2f}%")
            changed = True
        except Exception as e:
            print(f"  ⚠️  {r['name']} FP32 error: {e}")

    if int8_out.exists() and r["stm32_int8_acc_%"] is None:
        try:
            acc = compute_stm32_accuracy(SHARED_LABELS_NPZ, int8_out)
            r["stm32_int8_acc_%"] = round(acc, 2)
            print(f"  ✅ {r['name']:<42}  STM32 INT8: {acc:.2f}%")
            changed = True
        except Exception as e:
            print(f"  ⚠️  {r['name']} INT8 error: {e}")

    if changed:
        updated += 1

if updated > 0:
    with open(QUANTZ_RECORDS_JSON, "w") as f:
        json.dump(records, f, indent=2)
    print(f"\n💾 Saved {updated} updated records to {QUANTZ_RECORDS_JSON}")
else:
    print("No new STM32 results found. Copy output_fp32.npz / output_int8.npz")
    print("to their respective export dirs and re-run this cell.")

# Print full STM32 table
print(f"\n{'─'*80}")
print(f"{'Name':<42} {'FP32 val':>9} {'STM32 FP32':>11} {'STM32 INT8':>11} {'Delta':>8}")
print(f"{'─'*80}")
for r in records:
    fp32_stm = f"{r['stm32_fp32_acc_%']:.2f}%" if r["stm32_fp32_acc_%"] is not None else "pending"
    int8_stm = f"{r['stm32_int8_acc_%']:.2f}%" if r["stm32_int8_acc_%"] is not None else "pending"
    if r["stm32_fp32_acc_%"] and r["stm32_int8_acc_%"]:
        delta = f"{r['stm32_int8_acc_%'] - r['stm32_fp32_acc_%']:+.2f}%"
    else:
        delta = "—"
    print(f"{r['name']:<42} {r['fp32_val_acc_%']:>8.2f}% {fp32_stm:>11} {int8_stm:>11} {delta:>8}")

No new STM32 results found. Copy output_fp32.npz / output_int8.npz
to their respective export dirs and re-run this cell.

────────────────────────────────────────────────────────────────────────────────
Name                                        FP32 val  STM32 FP32  STM32 INT8    Delta
────────────────────────────────────────────────────────────────────────────────
mobilenetv2_seed_74                           78.40%     pending     pending        —
mobilenetv3_seed_74                           79.13%     pending     pending        —
mobilenetv2_unstructured_10pct                78.93%     pending     pending        —
mobilenetv2_unstructured_20pct                78.53%     pending     pending        —
mobilenetv2_unstructured_30pct                78.27%     pending     pending        —
mobilenetv3_unstructured_10pct                78.67%     pending     pending        —
mobilenetv3_unstructured_20pct                78.27%     pending     pending        —
mobilenetv3_unstructured_30p

In [ ]:
# ── STM32 deployment metrics (Flash / RAM / MACs) ────────────────────
# Run after stedgeai validate. Parses the analyze output for each model.
# stedgeai analyze -m exports/<name>/model_int8_qdq.onnx \
#                  --target stm32h7xx                     \
#                  --output exports/<name>/analyze_int8.txt

import re

def parse_stedgeai_analyze(txt_path):
    """Parse Flash, RAM, and MACC from stedgeai analyze output file."""
    if not Path(txt_path).exists():
        return None
    text = Path(txt_path).read_text()
    flash = re.search(r"Flash\s+\|\s+([\d,]+)\s+bytes", text)
    ram   = re.search(r"RAM\s+\|\s+([\d,]+)\s+bytes", text)
    macc  = re.search(r"MACCs\s+\|\s+([\d,]+)", text)
    return {
        "flash_kb": int(flash.group(1).replace(",","")) / 1024 if flash else None,
        "ram_kb":   int(ram.group(1).replace(",",""))   / 1024 if ram   else None,
        "macc_k":   int(macc.group(1).replace(",",""))  / 1000 if macc  else None,
    }

print(f"{'Model':<42} {'Flash(KB)':>10} {'RAM(KB)':>10} {'MACCs(K)':>10}")
print("-" * 75)
for r in quantz_records:
    analyze_path = Path(r["out_dir"]) / "analyze_int8.txt"
    m = parse_stedgeai_analyze(analyze_path)
    if m:
        flash_s = f"{m['flash_kb']:.1f}" if m["flash_kb"] else "—"
        ram_s   = f"{m['ram_kb']:.1f}"   if m["ram_kb"]   else "—"
        macc_s  = f"{m['macc_k']:.1f}"   if m["macc_k"]   else "—"
        print(f"  {r['name']:<40} {flash_s:>10} {ram_s:>10} {macc_s:>10}")
    else:
        print(f"  {r['name']:<40}   pending (run stedgeai analyze first)")

## Resume / re-run guide

**Add a new checkpoint:** just re-run `Model_Pruning_Combined` or `Model_KD_Combined` with the new experiment — the deployable list updates automatically in `pruning_combined_records.json` / `kd_combined_records.json`. Then re-run the build run list cell here and the export loop will pick up the new entry.

**Force re-export of one run:** delete its entry from `quantz_records.json` on Drive, delete its export folder under `exports/<name>/`, then re-run the export loop.

**Force everything from scratch:** delete `quantz_records.json` and re-run.

**STM32 results come in incrementally:** just copy each `output_fp32.npz` / `output_int8.npz` back to the right `exports/<name>/` folder and re-run Cell 12 — it only updates records that don't already have on-device results.